# سبق 04 - ٹول استعمال کرنے کا ڈیزائن پیٹرن

اس سبق میں آپ مائیکروسافٹ ایجنٹ فریم ورک (Python) استعمال کرتے ہوئے AI ایجنٹس کے لیے **ٹول استعمال** کے ڈیزائن پیٹرن سیکھیں گے۔ ہم مندرجہ ذیل موضوعات پر بات کریں گے:

- `@tool` ڈیکوریٹر اور ٹائپڈ پیرا میٹرز کے ساتھ فنکشن ٹولز کی تعریف کرنا
- ماڈل کو یہ معلوم ہونے کے لیے کہ ہر ٹول کیا کرتا ہے، ٹول اسکیمے فراہم کرنا
- `approval_mode` کے ذریعے ٹول کے نفاذ کو کنٹرول کرنا
- پائیڈانٹک ماڈلز اور `response_format` کے ذریعے **منظم آؤٹ پٹ** واپس کرنا

منظر نامہ ایک **سفر بکنگ ایجنٹ** ہے جو مقامات کی تلاش، دستیابی چیک کرنے، اور پرواز کی معلومات حاصل کر سکتا ہے۔


## سیٹ اپ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ڈیکوریٹر کے ساتھ ٹولز کی تعریف

`@tool` ڈیکوریٹر ایک سادہ پائتھن فنکشن کو ایک ایسے ٹول میں تبدیل کر دیتا ہے جسے ایجنٹ کال کر سکتا ہے۔  
اہم نکات:

- **ڈاکسٹرنگ** ٹول کی وضاحت بن جاتی ہے جو ماڈل دیکھتا ہے۔  
- **ٹائپ اینوٹیشنز** (بشمول وضاحتوں کے ساتھ `Annotated`) ٹول کا اسکیما متعین کرتے ہیں۔  
- `approval_mode` اس بات کو کنٹرول کرتا ہے کہ آیا صارف کو ہر کال کے اجرا سے پہلے منظوری دینی ہوگی۔


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## متعدد ٹولز کے ساتھ ایجنٹ بنانا

کلائنٹ کو تینوں ٹولز فراہم کریں تاکہ ماڈل صارف کے سوال کے جواب کے لیے جس کا بھی ضرورت ہو اسے استعمال کر سکے۔


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ٹولز کے ساتھ منظم آؤٹ پٹ

`response_format` کو Pydantic ماڈل پر سیٹ کرنے سے، ایجنٹ کو مجبور کیا جاتا ہے کہ وہ آزادانہ متن کی جگہ ایک اچھے طریقے سے ٹائپ کیا ہوا JSON آبجیکٹ واپس کرے۔ یہ اس وقت مفید ہے جب نیچے والا کوڈ نتیجہ پروگراماتی طور پر استعمال کرنا چاہتا ہو۔


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ٹول کی منظوری کے پیٹرنز

`@tool` پر `approval_mode` پیرا میٹر کنٹرول کرتا ہے کہ آیا ٹول کالز کو چلانے سے پہلے انسانی منظوری کی ضرورت ہے:

| موڈ | رویہ |
|---|---|
| `"never_require"` | ٹول خودکار طور پر چلتا ہے — صارف کی تصدیق کی ضرورت نہیں۔ |
| `"always_require"` | ہر کال کو چلانے سے پہلے صارف کی منظوری ضروری ہے۔ |

ایسے ٹولز کے لیے `"always_require"` استعمال کریں جن کے ضمنی اثرات ہوتے ہیں (مثلاً فلائٹ بک کرنا، کریڈٹ کارڈ چارج کرنا) تاکہ انسان اس عمل میں شامل رہے۔


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

1. **ٹولز کو متعین کریں** `@tool` ڈیکوریٹر کا استعمال کرتے ہوئے ٹائپ کیے گئے پیرامیٹرز اور ڈاک سٹرنگز کے ساتھ جو کہ ٹول اسکیمہ کے طور پر کام کرتی ہیں۔
2. **متعدد ٹولز کو ترتیب دیں** تاکہ ایجنٹ انہیں متسلسل طور پر کال کرکے پیچیدہ سوالات کے جوابات دے سکے۔
3. **منظم آؤٹ پٹ واپس کریں** ایک پائیڈانٹک ماڈل کو `response_format` کے طور پر پاس کرکے۔
4. **ٹول کی منظوری کو کنٹرول کریں** `approval_mode` کے ساتھ تاکہ حساس آپریشنز کے لیے انسان کو عمل کے دائرے میں رکھا جا سکے۔

یہ پیٹرنز قابل اعتماد، تیار شدہ پروڈکشن ایجنٹس بنانے کی بنیاد فراہم کرتے ہیں جو محفوظ طریقے سے بیرونی نظاموں کے ساتھ تعامل کر سکتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**دستخطی وضاحت**:  
اس دستاویز کا ترجمہ AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے کیا گیا ہے۔ اگرچہ ہم درستگی کی کوشش کرتے ہیں، براہ کرم آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا خامیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں مستند ماخذ سمجھا جانا چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کی ذمہ داری ہم پر نہیں ہوگی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
